# Assessment aligned example using Databricks

In [0]:
# Raw data stored in a volume
# load data

catalog_name = "workspace"
schema_name = "default"

business_path = "/Volumes/workspace/default/yelpdataset/yelp_academic_dataset_business.json"
review_path = "/Volumes/workspace/default/yelpdataset/yelp_academic_dataset_review.json"

spark.sql(f"USE CATALOG 'catalog_name'")

### Tip: Before defining the catalog name and the schema name, copy and paste the path, then check the path so that /Volumes/catalog_name/schema_name/.....

In [0]:
# Bronze layer

business_raw = spark.read.json(business_path) # read json into Spark
review_raw = spark.read.json(review_path)

# load bronze data into Delta tables

business_raw.write.mode("overwrite").format("delta").saveAsTable("bronze_business")
review_raw.write.mode("overwrite").format("delta").saveAsTable("bronze_review")

## Databricks is the workspace, Apache Spark is the engine

1. In the bronze level and after inserting json raw data into Delta tables, do I still have raw data?
> no, since the data are stored as Delta tables
1. What Delta tables are? 
> Delta table is a table stored using the Delta lake format, which adds structure on top of data files.


In [0]:
#  Silver layer

from pyspark.sql import functions as F

silver_business = business_raw.select(
    "business_id", "name", "state", "stars", "review_count", "categories"
).withColumnRenamed("stars", "business_stars")
silver_review = review_raw.select(
    "business_id", "stars", "date"
).withColumn("new_date", F.dayofmonth("date")).drop("date").withColumnRenamed("stars", "review_stars")

silver_business.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_business")
silver_review.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_review")


![image_1774688794933.png](./image_1774688794933.png "image_1774688794933.png")

### Trying to use from review_raw the date attribute, 

In [0]:
# Gold layer

gold_business_review = silver_review.join(silver_business, on="business_id", how="inner") # inner join keeps only the rows that match both tables

gold_summary = gold_business_review.groupBy("state").agg(
    F.countDistinct("business_id").alias("total_businesses"),
    F.countDistinct("review_count").alias("total_reviews"),
    F.round(F.avg("business_stars"),2).alias("avg_business_stars"),
    F.round(F.avg("review_stars"),2).alias("avg_review_stars")
)

gold_summary.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("gold_summary")

In [0]:
gold_summary.printSchema()
gold_summary.show(5)

In [0]:
silver_review.printSchema()
silver_review.show(5)

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW gold_summary AS
SELECT *
FROM gold_summary

In [0]:
%sql
SELECT 
    state,
    total_businesses,
    total_reviews,
    avg_business_stars,
    avg_review_stars
FROM gold_summary
ORDER BY total_reviews DESC 